## TRABAJO ATD COMPARADOR DE PRECIOS PS5
#### Hector Campos, Pablo Navarro, David Maudos


In [1]:
import re
import time
import webbrowser
import datetime

import json
import requests
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


archivo = 'comparador_precios_ps5.txt'
titulos = 'Fecha y Hora     | MediaMarkt | PC-Components | El Corte Inglés | GAME\n'
separador = "-" * 90 + "\n"

UA = {"User-Agent": "Mozilla/5.0"}

URLS = {
    "PlayStation": "https://www.playstation.com/es-es/ps5/",
    "MediaMarkt": "https://www.mediamarkt.es/es/product/_consola-sony-ps5-slim-digital-edition-825-gb-ssd-4k-1-mando-chasis-e-blanco-1605666.html",
    "PC-Components": "https://www.pccomponentes.com/sony-playstation-5-slim-digital-chasis-e",
    "El Corte Inglés": "https://www.elcorteingles.es/videojuegos/A200069099-consola-playstation-5-digital/?stype=text_box_multi&parentCategoryId=999.51648013&color=Sin+especificar",
    "GAME": "https://www.game.es/hardware/consola/playstation-5/playstation-5-modelo-slim-chassis-e/250405",
}


def _to_float_eur(texto: str):
    if not texto:
        return None
    s = texto.strip()

    # Extraemos el bloque numérico principal
    m = re.search(r'(\d{1,3}(?:[.\s]\d{3})*(?:[.,]\d{2})|\d+)', s)
    if not m:
        return None

    num = m.group(1)
    
    # CORRECCIÓN: Detección inteligente de formato
    # Si tiene coma, asumimos formato ES (1.000,00)
    if ',' in num:
        num = num.replace('.', '').replace(' ', '').replace("'", "")
        num = num.replace(',', '.')
    elif '.' in num:
        # Si tiene punto pero NO coma:
        # - Caso 1: "499.00" (Typical JSON/Float) -> Dejar el punto.
        # - Caso 2: "1.200" (Miles visual) -> Este es el conflicto.
        # Dado que el error reportado es 499.00 -> 49900, asumimos formato punto-decimal
        # si parece un número decimal estándar (2 decimales) o si viene de metadatos.
        pass
    else:
        # Sin puntos ni comas, nada que limpiar
        pass
        
    try:
        return float(num)
    except:
        return None

def _extract_price_from_html(html: str):
    """
    Intenta sacar el precio de forma robusta desde el HTML (meta tags / itemprop).
    """
    soup = BeautifulSoup(html, "html.parser")

    meta = soup.select_one('meta[itemprop="price"]')
    if meta and meta.get("content"):
        return meta["content"]

    meta = soup.select_one('meta[property="product:price:amount"]')
    if meta and meta.get("content"):
        return meta["content"]

    meta = soup.select_one('meta[property="og:price:amount"]')
    if meta and meta.get("content"):
        return meta["content"]

    node = soup.select_one('[itemprop="price"]')
    if node:
        if node.get("content"):
            return node["content"]
        txt = node.get_text(" ", strip=True)
        if txt:
            return txt
    for sc in soup.select('script[type="application/ld+json"]'):
        try:
            raw_json = sc.string or sc.get_text()
            if not raw_json:
                continue
            data = json.loads(raw_json)
            items = data if isinstance(data, list) else [data]
            for it in items:
                if not isinstance(it, dict):
                    continue
                offers = it.get("offers")
                if isinstance(offers, list) and offers:
                    offers = offers[0]
                if isinstance(offers, dict):
                    price = offers.get("price") or offers.get("lowPrice") or offers.get("highPrice")
                    if price is not None:
                        return str(price)
        except Exception:
            pass


    return None


def _get_page_html_selenium(url: str, wait_css: str = None, timeout: int = 20):
    driver = webdriver.Chrome()
    try:
        driver.get(url)
        if wait_css:
            WebDriverWait(driver, timeout).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, wait_css))
            )
        time.sleep(1.2)
        return driver.page_source
    finally:
        driver.quit()
        
def get_model():
    html = _get_page_html_selenium(URLS["PlayStation"], wait_css="h1")
    soup = BeautifulSoup(html, "html.parser")
    h1 = soup.find("h1")
    if h1 and h1.get_text(strip=True):
        return h1.get_text(strip=True)
    return "PlayStation 5"


def get_price_mediamarkt():
    html = _get_page_html_selenium(URLS["MediaMarkt"])
    raw = _extract_price_from_html(html)
    if not raw:
        m = re.search(r'(\d[\d\.,\s]{1,12})\s*€', html)
        raw = m.group(0) if m else None
    p = _to_float_eur(raw)
    return p


def get_price_pc_components():
    html = _get_page_html_selenium(URLS["PC-Components"], wait_css="script[type='application/ld+json']")
    raw = _extract_price_from_html(html)
    if not raw:
        m = re.search(r'(\d[\d\.,\s]{1,12})\s*€', html)
        raw = m.group(0) if m else None
    p = _to_float_eur(raw)
    return p


def get_price_elcorteingles():
    html = _get_page_html_selenium(URLS["El Corte Inglés"], wait_css="script[type='application/ld+json']")
    raw = _extract_price_from_html(html)
    if not raw:
        m = re.search(r'(\d[\d\.,\s]{1,12})\s*€', html)
        raw = m.group(0) if m else None
    p = _to_float_eur(raw)
    return p


def get_price_game():
    html = _get_page_html_selenium(URLS["GAME"])
    raw = _extract_price_from_html(html)
    if not raw:
        m = re.search(r'(\d[\d\.,\s]{1,12})\s*€', html)
        raw = m.group(0) if m else None
    p = _to_float_eur(raw)
    return p

def main():
    modelo = get_model()

    precios_float = {
        'MediaMarkt': get_price_mediamarkt(),
        'PC-Components': get_price_pc_components(),
        'El Corte Inglés': get_price_elcorteingles(),
        'GAME': get_price_game()
    }

    def fmt(p):
        return (f"{p:.2f}€" if isinstance(p, (int, float)) else "N/D")

    fecha = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    linea = (
        f"{fecha} | {fmt(precios_float['MediaMarkt'])} | {fmt(precios_float['PC-Components'])} | "
        f"{fmt(precios_float['El Corte Inglés'])} | {fmt(precios_float['GAME'])}\n"
    )

    try:
        with open(archivo, 'r', encoding="utf-8") as f:
            contenido = f.read()
            if contenido.strip() == '':
                raise FileNotFoundError
    except FileNotFoundError:
        with open(archivo, 'w', encoding="utf-8") as f:
            f.write(f"Modelo: {modelo}\n\n")
            f.write(titulos)
            f.write(separador)

    with open(archivo, 'a', encoding="utf-8") as f:
        f.write(linea)
        f.write(separador)
        
    disponibles = [(tienda, p) for tienda, p in precios_float.items() if isinstance(p, (int, float))]
    if disponibles:
        mejor_tienda, mejor_precio = min(disponibles, key=lambda x: x[1])
        print(f"Mejor precio: {mejor_tienda} -> {mejor_precio:.2f}€")
        if mejor_tienda in URLS:
            webbrowser.open(URLS[mejor_tienda])
    else:
        print("No se ha podido detectar ningún precio")
        
if __name__ == '__main__':
    main()


Mejor precio: PC-Components -> 415.00€
